# Bronze Layer

### Objective

Load the raw datasets into the Bronze layer while preserving the original data._

# Bronze Layer

## Objective

Load the ingested customer support datasets into the Bronze layer while preserving the original data and adding ingestion metadata for tracking.

Input datasets:
- agent_profiles
- day1_tickets
- day2_tickets

Output:
- bronze_profiles
- bronze_day1
- bronze_day2

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

CATALOG = "workspace"
SCHEMA = "hrm6131_landing"

# Raw Tables
TBL_PROFILES = f"{CATALOG}.{SCHEMA}.agent_profiles"
TBL_DAY1 = f"{CATALOG}.{SCHEMA}.day1_tickets"
TBL_DAY2 = f"{CATALOG}.{SCHEMA}.day2_tickets"

# Bronze Tables
BRONZE_PROFILES = f"{CATALOG}.{SCHEMA}.bronze_profiles"
BRONZE_DAY1 = f"{CATALOG}.{SCHEMA}.bronze_day1"
BRONZE_DAY2 = f"{CATALOG}.{SCHEMA}.bronze_day2"

print("Configuration loaded.")

## Read Raw Tables

In [0]:
raw_profiles = spark.table(TBL_PROFILES)
raw_day1 = spark.table(TBL_DAY1)
raw_day2 = spark.table(TBL_DAY2)

In [0]:
display(raw_profiles)
display(raw_day1)
display(raw_day2)

## Add Ingestion Metadata

In [0]:
bronze_profiles = (
    raw_profiles
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_file", F.lit("agent_profiles.csv"))
)

In [0]:
bronze_day1 = (
    raw_day1
    .withColumnRenamed("category", "ticket_category")
    .withColumn("day", F.lit(1))
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_file", F.lit("day1_tickets.csv"))
)

In [0]:
bronze_day2 = (
    raw_day2
    .withColumn("day", F.lit(2))
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_file", F.lit("day2_tickets.csv"))
)

## Save Bronze Tables

In [0]:
(
    bronze_profiles.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_PROFILES)
)

(
    bronze_day1.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_DAY1)
)

(
    bronze_day2.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_DAY2)
)

print("Bronze tables created successfully.")

## Validate Bronze Layer

In [0]:
print(f"Bronze Profiles : {spark.table(BRONZE_PROFILES).count()} rows")
print(f"Bronze Day 1    : {spark.table(BRONZE_DAY1).count()} rows")
print(f"Bronze Day 2    : {spark.table(BRONZE_DAY2).count()} rows")

In [0]:
display(spark.table(BRONZE_PROFILES))
display(spark.table(BRONZE_DAY1))
display(spark.table(BRONZE_DAY2))